# Dynamic ADT Scene Scanpath 3D Animation

This notebook builds a Plotly animation for ADT scene-space scanpath comparison. It follows the same data logic as `04_scene_scanpath_3d.py`:

- GT gaze points are read from `gaze_samples.csv`.
- SparseGaze and HAGI++ provide directions; 3D endpoints are placed at aligned GT depth.
- Scene context can include object boxes, head/body trajectory, current skeleton, and GT ray-box hit.

Unlike `11_scene_scanpath_3d_interactive.ipynb`, this notebook precomputes Plotly `frames` and provides Play/Pause controls inside the 3D figure.

In [1]:
from __future__ import annotations

import importlib.util
import sys
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import clear_output, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'Experiments').exists():
    REPO_ROOT = Path('/home/liumu/Github_Projects/adt_dataset_sandbox')

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

SCENE04_PATH = REPO_ROOT / 'Experiments' / 'visualization & Analysis' / 'ADT' / '04_scene_scanpath_3d.py'
spec = importlib.util.spec_from_file_location('scene_scanpath_04', SCENE04_PATH)
scene04 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = scene04
spec.loader.exec_module(scene04)

REPORTS_DIR = Path('/mnt/d/SparseGaze/ADT-Gaze-structured')
SPARSEGAZE_SEQUENCE_ROOT = Path('/home/liumu/Github_Projects/SparseGaze/outputs/eval/adt/sparsegaze/test/rollout/sequence_predictions')
HAGI_NPZ = Path('/home/liumu/Github_Projects/HAGI/save/head/hagi++_imputation/adt_low_framerate_sliding/sliding_primary_nsample20_framerate_6.npz')
HAGI_ADT_DATA = Path('/home/liumu/Github_Projects/HAGI/datasets/adt/gaze_head_adt.npy')

TRACK_LABELS = ['GT', 'SparseGaze', 'HAGI++']
TRACK_COLORS = scene04.TRACK_COLORS
BOX_EDGES = scene04.BOX_EDGES


In [2]:
def list_sequences(reports_dir: Path = REPORTS_DIR) -> list[str]:
    base = reports_dir / 'sequences'
    if not base.exists():
        return []
    return sorted(path.name for path in base.iterdir() if path.is_dir())


def sparsegaze_npz_for_sequence(sequence: str) -> Path:
    return SPARSEGAZE_SEQUENCE_ROOT / sequence / 'hz6_phase0.npz'


@lru_cache(maxsize=8)
def load_sequence_data(sequence: str, use_sparsegaze: bool = True, use_hagi: bool = True):
    context = scene04.load_scene_context(REPORTS_DIR, sequence)
    prediction_npz = sparsegaze_npz_for_sequence(sequence) if use_sparsegaze else None
    hagi_npz = HAGI_NPZ if use_hagi else None
    try:
        tracks, table = scene04.build_track_table(
            context=context,
            prediction_npz=prediction_npz,
            pred_key='pred_xyz',
            hagi_npz=hagi_npz,
            hagi_adt_data=HAGI_ADT_DATA,
        )
    except Exception as exc:
        print(f'warning: failed to load HAGI++ for {sequence}: {exc}')
        tracks, table = scene04.build_track_table(
            context=context,
            prediction_npz=prediction_npz,
            pred_key='pred_xyz',
            hagi_npz=None,
            hagi_adt_data=HAGI_ADT_DATA,
        )
    return context, tracks, table


def sample_rows(rows: pd.DataFrame, stride: int) -> pd.DataFrame:
    stride = max(int(stride), 1)
    if len(rows) <= 1 or stride == 1:
        return rows
    sampled = rows.iloc[::stride].copy()
    if int(sampled['frame_index'].iloc[-1]) != int(rows['frame_index'].iloc[-1]):
        sampled = pd.concat([sampled, rows.tail(1)], ignore_index=True)
    return sampled


def display_points(points: np.ndarray, vertical_axis: str, mirror_horizontal: bool) -> np.ndarray:
    return scene04.display_points(points, vertical_axis, mirror_horizontal)


def finite_xyz(frame: pd.DataFrame, columns: list[str]) -> np.ndarray:
    return scene04.finite_xyz(frame, columns)


def empty_trace(name: str, showlegend: bool = True):
    return go.Scatter3d(x=[], y=[], z=[], mode='markers', name=name, showlegend=showlegend)


def filter_objects(objects: pd.DataFrame | None, max_static_objects: int, exclude_category_filter: str = 'shelter'):
    if objects is None or objects.empty:
        return None, None
    excluded = scene04.parse_filter(exclude_category_filter)
    static_rows = objects[objects['timestamp_ns'] == -1].copy()
    dynamic_rows = objects[objects['timestamp_ns'] != -1].copy()
    static_rows = scene04.filter_object_categories(static_rows, None, excluded)
    dynamic_rows = scene04.filter_object_categories(dynamic_rows, None, excluded)
    if max_static_objects > 0:
        static_rows = static_rows.head(max_static_objects)
    return static_rows, dynamic_rows


def nearest_dynamic_rows(dynamic_rows: pd.DataFrame | None, timestamp_ns: int) -> pd.DataFrame | None:
    if dynamic_rows is None or dynamic_rows.empty:
        return None
    timestamps = dynamic_rows['timestamp_ns'].to_numpy(dtype=np.int64)
    nearest = int(timestamps[np.argmin(np.abs(timestamps - int(timestamp_ns)))])
    return dynamic_rows[dynamic_rows['timestamp_ns'] == nearest]


In [3]:
def box_trace(rows, name, color, opacity, vertical_axis, mirror_horizontal, showlegend=True):
    if rows is None or rows.empty:
        return empty_trace(name, showlegend=showlegend)
    xs, ys, zs = [], [], []
    for _, row in rows.iterrows():
        corners = display_points(scene04.row_corners(row), vertical_axis, mirror_horizontal)
        for first, second in BOX_EDGES:
            xs.extend([corners[first, 0], corners[second, 0], None])
            ys.extend([corners[first, 1], corners[second, 1], None])
            zs.extend([corners[first, 2], corners[second, 2], None])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', name=name, line=dict(color=color, width=2), opacity=opacity, hoverinfo='skip', showlegend=showlegend)


def track_tail_trace(rows, track, enabled_tracks, vertical_axis, mirror_horizontal):
    if track not in enabled_tracks:
        return empty_trace(f'{track} scanpath')
    prefix = scene04.track_prefix(track)
    points = finite_xyz(rows, [f'{prefix}_x', f'{prefix}_y', f'{prefix}_z'])
    if len(points) == 0:
        return empty_trace(f'{track} scanpath')
    points = display_points(points, vertical_axis, mirror_horizontal)
    color = TRACK_COLORS.get(track, '#111111')
    return go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode='lines+markers', name=f'{track} scanpath',
        line=dict(color=color, width=7), marker=dict(size=4, color=color),
    )


def track_current_trace(rows, track, enabled_tracks, current_frame, vertical_axis, mirror_horizontal):
    if track not in enabled_tracks:
        return empty_trace(f'{track} current', showlegend=False)
    prefix = scene04.track_prefix(track)
    current = rows[rows['frame_index'] == current_frame]
    points = finite_xyz(current, [f'{prefix}_x', f'{prefix}_y', f'{prefix}_z'])
    if len(points) == 0:
        return empty_trace(f'{track} current', showlegend=False)
    points = display_points(points, vertical_axis, mirror_horizontal)
    color = TRACK_COLORS.get(track, '#111111')
    return go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode='markers', name=f'{track} current',
        marker=dict(size=8, color=color, symbol='diamond'), showlegend=False,
    )


def head_trace(head, rows, current_frame, vertical_axis, mirror_horizontal, show_head):
    if not show_head or head is None or head.empty:
        return empty_trace('head/device trajectory')
    timestamps = set(rows[rows['frame_index'] <= current_frame]['query_timestamp_ns'].astype(np.int64))
    head_rows = head[head['query_timestamp_ns'].isin(timestamps)]
    points = finite_xyz(head_rows, ['head_origin_scene_x_m', 'head_origin_scene_y_m', 'head_origin_scene_z_m'])
    if len(points) == 0:
        return empty_trace('head/device trajectory')
    points = display_points(points, vertical_axis, mirror_horizontal)
    return go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode='lines', name='head/device trajectory', line=dict(color='#0f766e', width=5))


def body_trace(skeleton, rows, current_frame, vertical_axis, mirror_horizontal, show_body):
    if not show_body or skeleton is None or skeleton.empty or 'frame_index' not in skeleton.columns:
        return empty_trace('body/root trajectory')
    frame_set = set(rows[rows['frame_index'] <= current_frame]['frame_index'].astype(int))
    skel_rows = skeleton[skeleton['frame_index'].astype(int).isin(frame_set)]
    points = finite_xyz(skel_rows, ['root_joint_scene_x_m', 'root_joint_scene_y_m', 'root_joint_scene_z_m'])
    if len(points) == 0:
        return empty_trace('body/root trajectory')
    points = display_points(points, vertical_axis, mirror_horizontal)
    return go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode='lines', name='body/root trajectory', line=dict(color='#9333ea', width=4))


def skeleton_traces(skeleton, skeleton_summary, current_frame, vertical_axis, mirror_horizontal, show_skeleton):
    if not show_skeleton or skeleton is None or skeleton.empty or not skeleton_summary or 'frame_index' not in skeleton.columns:
        return empty_trace('current skeleton'), empty_trace('skeleton joints', showlegend=False)
    nearest = skeleton.iloc[int(np.argmin(np.abs(skeleton['frame_index'].to_numpy(dtype=int) - current_frame)))]
    labels = skeleton_summary.get('joint_labels', [])
    connections = skeleton_summary.get('joint_connections', [])
    points = []
    for joint_index, label in enumerate(labels):
        safe = scene04.safe_joint_label(str(label))
        columns = [f'joint_{joint_index:02d}_{safe}_scene_x_m', f'joint_{joint_index:02d}_{safe}_scene_y_m', f'joint_{joint_index:02d}_{safe}_scene_z_m']
        if not all(column in nearest.index for column in columns):
            points.append([np.nan, np.nan, np.nan])
        else:
            points.append([float(nearest[column]) for column in columns])
    point_array = np.asarray(points, dtype=np.float64)
    finite = np.isfinite(point_array).all(axis=1)
    if not finite.any():
        return empty_trace('current skeleton'), empty_trace('skeleton joints', showlegend=False)
    point_array = display_points(point_array, vertical_axis, mirror_horizontal)
    xs, ys, zs = [], [], []
    for first_joint, second_joint in connections:
        if first_joint >= len(point_array) or second_joint >= len(point_array):
            continue
        if not (finite[first_joint] and finite[second_joint]):
            continue
        xs.extend([point_array[first_joint, 0], point_array[second_joint, 0], None])
        ys.extend([point_array[first_joint, 1], point_array[second_joint, 1], None])
        zs.extend([point_array[first_joint, 2], point_array[second_joint, 2], None])
    line_trace = go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', name='current skeleton', line=dict(color='#7c3aed', width=4))
    joint_trace = go.Scatter3d(x=point_array[finite, 0], y=point_array[finite, 1], z=point_array[finite, 2], mode='markers', name='skeleton joints', marker=dict(color='#a78bfa', size=3), showlegend=False)
    return line_trace, joint_trace


def hit_trace(window, current_frame, vertical_axis, mirror_horizontal, show_hit):
    if not show_hit:
        return empty_trace('hit point')
    current = window[window['frame_index'] == current_frame]
    if current.empty:
        return empty_trace('hit point')
    row = current.iloc[-1]
    if not bool(row.get('object_hit', False)):
        return empty_trace('hit point')
    point = np.asarray([row.get('hit_x_m'), row.get('hit_y_m'), row.get('hit_z_m')], dtype=float)
    if not np.isfinite(point).all():
        return empty_trace('hit point')
    point = display_points(point.reshape(1, 3), vertical_axis, mirror_horizontal)[0]
    label = row.get('hit_instance_name', 'GT ray-box hit')
    return go.Scatter3d(x=[point[0]], y=[point[1]], z=[point[2]], mode='markers', name=f'hit: {label}', marker=dict(color='#dc2626', size=8, symbol='diamond'))


In [4]:
def choose_animation_frames(window: pd.DataFrame, frame_stride: int, max_frames: int) -> list[int]:
    frames = window['frame_index'].to_numpy(dtype=int)
    selected = list(frames[::max(int(frame_stride), 1)])
    if len(frames) and selected[-1] != int(frames[-1]):
        selected.append(int(frames[-1]))
    max_frames = max(int(max_frames), 1)
    if len(selected) > max_frames:
        indices = np.linspace(0, len(selected) - 1, max_frames).round().astype(int)
        selected = [selected[i] for i in sorted(set(indices))]
        if selected[-1] != int(frames[-1]):
            selected.append(int(frames[-1]))
    return [int(frame) for frame in selected]


def frame_trace_bundle(
    *,
    context,
    window,
    static_rows,
    dynamic_rows,
    tracks,
    enabled_tracks,
    current_frame,
    point_stride,
    tail_frames,
    show_static_objects,
    show_dynamic_objects,
    show_head,
    show_body,
    show_skeleton,
    show_hit,
    vertical_axis,
    mirror_horizontal,
):
    focus_rows = window[window['frame_index'] <= current_frame]
    if focus_rows.empty:
        focus_rows = window.head(1)
    focus_timestamp = int(focus_rows['query_timestamp_ns'].iloc[-1])
    tail = focus_rows[focus_rows['frame_index'] >= current_frame - int(tail_frames)].copy()
    tail = sample_rows(tail, point_stride)
    trajectory_rows = sample_rows(focus_rows, point_stride)
    current_dynamic_rows = nearest_dynamic_rows(dynamic_rows, focus_timestamp)

    traces = [
        box_trace(static_rows if show_static_objects else None, 'static object boxes', '#6b7280', 0.22, vertical_axis, mirror_horizontal),
        box_trace(current_dynamic_rows if show_dynamic_objects else None, 'dynamic object boxes', '#2563eb', 0.55, vertical_axis, mirror_horizontal),
        head_trace(context.head, trajectory_rows, current_frame, vertical_axis, mirror_horizontal, show_head),
        body_trace(context.skeleton, trajectory_rows, current_frame, vertical_axis, mirror_horizontal, show_body),
    ]
    skel_line, skel_points = skeleton_traces(context.skeleton, context.skeleton_summary, current_frame, vertical_axis, mirror_horizontal, show_skeleton)
    traces.extend([skel_line, skel_points])
    for track in TRACK_LABELS:
        if track in tracks:
            traces.append(track_tail_trace(tail, track, enabled_tracks, vertical_axis, mirror_horizontal))
            traces.append(track_current_trace(tail, track, enabled_tracks, current_frame, vertical_axis, mirror_horizontal))
        else:
            traces.append(empty_trace(f'{track} scanpath'))
            traces.append(empty_trace(f'{track} current', showlegend=False))
    traces.append(hit_trace(window, current_frame, vertical_axis, mirror_horizontal, show_hit))
    return traces


def axis_ranges_for_animation(frames_data):
    plotted = []
    for traces in frames_data:
        for trace in traces:
            x = np.asarray(trace.x if trace.x is not None else [], dtype=object)
            y = np.asarray(trace.y if trace.y is not None else [], dtype=object)
            z = np.asarray(trace.z if trace.z is not None else [], dtype=object)
            if len(x) == 0:
                continue
            values = []
            for xi, yi, zi in zip(x, y, z):
                if xi is None or yi is None or zi is None:
                    continue
                try:
                    values.append([float(xi), float(yi), float(zi)])
                except (TypeError, ValueError):
                    continue
            if values:
                plotted.append(np.asarray(values, dtype=np.float64))
    return scene04.axis_ranges(plotted)


def nearest_head_pose_row(head: pd.DataFrame | None, timestamp_ns: int) -> pd.Series | None:
    if head is None or head.empty or 'query_timestamp_ns' not in head.columns:
        return None
    timestamps = head['query_timestamp_ns'].to_numpy(dtype=np.int64)
    if len(timestamps) == 0:
        return None
    return head.iloc[int(np.argmin(np.abs(timestamps - int(timestamp_ns))))]


def scene_to_cpf_directions(rows: pd.DataFrame, head: pd.DataFrame | None, prefix: str) -> np.ndarray:
    cols = [f'{prefix}_dir_x', f'{prefix}_dir_y', f'{prefix}_dir_z']
    output = np.full((len(rows), 3), np.nan, dtype=np.float64)
    if not all(col in rows.columns for col in cols):
        return output
    for row_pos, row in enumerate(rows.itertuples(index=False)):
        direction = np.asarray(
            [getattr(row, cols[0]), getattr(row, cols[1]), getattr(row, cols[2])],
            dtype=np.float64,
        )
        if not np.isfinite(direction).all() or np.linalg.norm(direction) <= 1e-8:
            continue
        head_row = nearest_head_pose_row(head, int(getattr(row, 'query_timestamp_ns')))
        if head_row is None:
            continue
        right = np.asarray([head_row.get('head_right_scene_unit_x'), head_row.get('head_right_scene_unit_y'), head_row.get('head_right_scene_unit_z')], dtype=np.float64)
        up = np.asarray([head_row.get('head_up_scene_unit_x'), head_row.get('head_up_scene_unit_y'), head_row.get('head_up_scene_unit_z')], dtype=np.float64)
        forward = np.asarray([head_row.get('head_forward_scene_unit_x'), head_row.get('head_forward_scene_unit_y'), head_row.get('head_forward_scene_unit_z')], dtype=np.float64)
        if not (np.isfinite(right).all() and np.isfinite(up).all() and np.isfinite(forward).all()):
            continue
        r_scene_cpf = np.column_stack([right, up, forward])
        output[row_pos] = r_scene_cpf.T @ direction
    return output


def direction_pitch_yaw_deg(rows: pd.DataFrame, head: pd.DataFrame | None, prefix: str) -> tuple[np.ndarray, np.ndarray]:
    dirs = scene_to_cpf_directions(rows, head, prefix)
    valid = np.isfinite(dirs).all(axis=1) & (np.linalg.norm(dirs, axis=1) > 1e-8)
    x = dirs[:, 0]
    y = dirs[:, 1]
    z = dirs[:, 2]
    yaw = np.degrees(np.arctan2(x, z))
    pitch = np.degrees(np.arctan2(y, np.sqrt(x * x + z * z)))
    yaw[~valid] = np.nan
    pitch[~valid] = np.nan
    return pitch, yaw


def diagnostic_rows_for_frames(window: pd.DataFrame, animation_frames: list[int]) -> pd.DataFrame:
    rows = window[window['frame_index'].isin(animation_frames)].copy()
    frame_order = {frame: i for i, frame in enumerate(animation_frames)}
    rows['_frame_order'] = rows['frame_index'].map(frame_order)
    return rows.sort_values('_frame_order').drop(columns=['_frame_order'])


def diagnostic_value(diag_rows: pd.DataFrame, frame_index: int, column: str) -> float:
    if column not in diag_rows.columns:
        return np.nan
    row = diag_rows[diag_rows['frame_index'] == frame_index]
    if row.empty:
        return np.nan
    return float(row[column].iloc[-1])


def diagnostic_angle_value(diag_rows: pd.DataFrame, frame_index: int, values: np.ndarray) -> float:
    matches = np.flatnonzero(diag_rows['frame_index'].to_numpy(dtype=int) == int(frame_index))
    if len(matches) == 0:
        return np.nan
    return float(values[matches[-1]])


def break_angle_wraps(frames: np.ndarray, values: np.ndarray, threshold_deg: float = 180.0) -> tuple[list[float], list[float]]:
    out_x: list[float] = []
    out_y: list[float] = []
    prev = np.nan
    for frame, value in zip(frames, values):
        if np.isfinite(prev) and np.isfinite(value) and abs(float(value) - float(prev)) > threshold_deg:
            out_x.append(np.nan)
            out_y.append(np.nan)
        out_x.append(float(frame))
        out_y.append(float(value) if np.isfinite(value) else np.nan)
        prev = value
    return out_x, out_y


def clipped_angle_axis_range(
    angle_cache: dict[str, tuple[np.ndarray, np.ndarray]],
    angle_index: int,
    enabled_tracks: tuple[str, ...],
    bounds: tuple[float, float],
    min_span_deg: float = 20.0,
    pad_fraction: float = 0.10,
) -> list[float]:
    track_prefixes = {'GT': 'gt', 'SparseGaze': 'sparsegaze', 'HAGI++': 'hagi'}
    arrays = []
    for label, prefix in track_prefixes.items():
        if label not in enabled_tracks or prefix not in angle_cache:
            continue
        values = np.asarray(angle_cache[prefix][angle_index], dtype=float)
        values = values[np.isfinite(values)]
        if len(values):
            arrays.append(values)
    if not arrays:
        return [float(bounds[0]), float(bounds[1])]

    values = np.concatenate(arrays)
    lo = float(np.nanmin(values))
    hi = float(np.nanmax(values))
    center = 0.5 * (lo + hi)
    span = max(hi - lo, min_span_deg)
    span *= 1.0 + 2.0 * pad_fraction

    lo = center - 0.5 * span
    hi = center + 0.5 * span
    bound_lo, bound_hi = map(float, bounds)
    if lo < bound_lo:
        hi = min(bound_hi, hi + (bound_lo - lo))
        lo = bound_lo
    if hi > bound_hi:
        lo = max(bound_lo, lo - (hi - bound_hi))
        hi = bound_hi
    return [float(lo), float(hi)]


def diagnostic_static_traces(diag_rows: pd.DataFrame, head: pd.DataFrame | None, enabled_tracks: tuple[str, ...]):
    frames = diag_rows['frame_index'].to_numpy(dtype=int)
    traces = []
    for prefix, label, color in [
        ('sparsegaze', 'SparseGaze angular error', TRACK_COLORS['SparseGaze']),
        ('hagi', 'HAGI++ angular error', TRACK_COLORS['HAGI++']),
    ]:
        y = diag_rows[f'{prefix}_angular_error_deg'].to_numpy(dtype=float) if f'{prefix}_angular_error_deg' in diag_rows.columns and label.split()[0] in enabled_tracks else np.full(len(diag_rows), np.nan)
        traces.append(go.Scatter(x=frames, y=y, mode='lines+markers', name=label, line=dict(color=color), marker=dict(size=5)))
    for prefix, label, color in [
        ('sparsegaze', 'SparseGaze Euclidean distance', TRACK_COLORS['SparseGaze']),
        ('hagi', 'HAGI++ Euclidean distance', TRACK_COLORS['HAGI++']),
    ]:
        y = diag_rows[f'{prefix}_point_error_m'].to_numpy(dtype=float) if f'{prefix}_point_error_m' in diag_rows.columns and label.split()[0] in enabled_tracks else np.full(len(diag_rows), np.nan)
        traces.append(go.Scatter(x=frames, y=y, mode='lines+markers', name=label, line=dict(color=color), marker=dict(size=5)))

    angle_cache = {}
    for prefix, label in [('gt', 'GT'), ('sparsegaze', 'SparseGaze'), ('hagi', 'HAGI++')]:
        angle_cache[prefix] = direction_pitch_yaw_deg(diag_rows, head, prefix)
    for prefix, label, color in [('gt', 'GT', TRACK_COLORS['GT']), ('sparsegaze', 'SparseGaze', TRACK_COLORS['SparseGaze']), ('hagi', 'HAGI++', TRACK_COLORS['HAGI++'])]:
        pitch, _ = angle_cache[prefix]
        if label not in enabled_tracks:
            pitch = np.full(len(diag_rows), np.nan)
        traces.append(go.Scatter(x=frames, y=pitch, mode='lines+markers', name=f'{label} pitch', line=dict(color=color), marker=dict(size=5)))
    for prefix, label, color in [('gt', 'GT', TRACK_COLORS['GT']), ('sparsegaze', 'SparseGaze', TRACK_COLORS['SparseGaze']), ('hagi', 'HAGI++', TRACK_COLORS['HAGI++'])]:
        _, yaw = angle_cache[prefix]
        if label not in enabled_tracks:
            yaw = np.full(len(diag_rows), np.nan)
        yaw_x, yaw_y = break_angle_wraps(frames, yaw)
        traces.append(go.Scatter(x=yaw_x, y=yaw_y, mode='lines+markers', name=f'{label} yaw', line=dict(color=color), marker=dict(size=5)))
    return traces, angle_cache


def diagnostic_marker_traces(diag_rows: pd.DataFrame, frame_index: int, enabled_tracks: tuple[str, ...], angle_cache: dict):
    traces = []
    for prefix, label, color in [('sparsegaze', 'SparseGaze', TRACK_COLORS['SparseGaze']), ('hagi', 'HAGI++', TRACK_COLORS['HAGI++'])]:
        y = diagnostic_value(diag_rows, frame_index, f'{prefix}_angular_error_deg') if label in enabled_tracks else np.nan
        traces.append(go.Scatter(x=[frame_index], y=[y], mode='markers', name=f'{label} current angular error', marker=dict(color=color, size=10, symbol='diamond'), showlegend=False))
    for prefix, label, color in [('sparsegaze', 'SparseGaze', TRACK_COLORS['SparseGaze']), ('hagi', 'HAGI++', TRACK_COLORS['HAGI++'])]:
        y = diagnostic_value(diag_rows, frame_index, f'{prefix}_point_error_m') if label in enabled_tracks else np.nan
        traces.append(go.Scatter(x=[frame_index], y=[y], mode='markers', name=f'{label} current distance', marker=dict(color=color, size=10, symbol='diamond'), showlegend=False))
    for prefix, label, color in [('gt', 'GT', TRACK_COLORS['GT']), ('sparsegaze', 'SparseGaze', TRACK_COLORS['SparseGaze']), ('hagi', 'HAGI++', TRACK_COLORS['HAGI++'])]:
        pitch, _ = angle_cache[prefix]
        y = diagnostic_angle_value(diag_rows, frame_index, pitch) if label in enabled_tracks else np.nan
        traces.append(go.Scatter(x=[frame_index], y=[y], mode='markers', name=f'{label} current pitch', marker=dict(color=color, size=10, symbol='diamond'), showlegend=False))
    for prefix, label, color in [('gt', 'GT', TRACK_COLORS['GT']), ('sparsegaze', 'SparseGaze', TRACK_COLORS['SparseGaze']), ('hagi', 'HAGI++', TRACK_COLORS['HAGI++'])]:
        _, yaw = angle_cache[prefix]
        y = diagnostic_angle_value(diag_rows, frame_index, yaw) if label in enabled_tracks else np.nan
        traces.append(go.Scatter(x=[frame_index], y=[y], mode='markers', name=f'{label} current yaw', marker=dict(color=color, size=10, symbol='diamond'), showlegend=False))
    return traces


def build_animation_scene_figure(
    sequence: str,
    start_frame: int,
    end_frame: int,
    frame_stride: int,
    point_stride: int,
    tail_frames: int,
    max_animation_frames: int,
    enabled_tracks: tuple[str, ...],
    show_static_objects: bool,
    show_dynamic_objects: bool,
    show_head: bool,
    show_body: bool,
    show_skeleton: bool,
    show_hit: bool,
    max_static_objects: int,
    vertical_axis: str,
    mirror_horizontal: bool,
    view_elev: float,
    view_azim: float,
    frame_duration_ms: int,
):
    context, tracks, table = load_sequence_data(sequence, 'SparseGaze' in enabled_tracks, 'HAGI++' in enabled_tracks)
    start_frame = max(0, int(start_frame))
    end_frame = min(int(end_frame), int(table['frame_index'].max()) + 1)
    if end_frame <= start_frame:
        end_frame = start_frame + 1
    window = scene04.select_window(table, start_frame, end_frame)
    animation_frames = choose_animation_frames(window, frame_stride, max_animation_frames)
    static_rows, dynamic_rows = filter_objects(context.objects, max_static_objects)

    scene_frame_data = []
    for frame_index in animation_frames:
        traces = frame_trace_bundle(
            context=context,
            window=window,
            static_rows=static_rows,
            dynamic_rows=dynamic_rows,
            tracks=tracks,
            enabled_tracks=enabled_tracks,
            current_frame=int(frame_index),
            point_stride=point_stride,
            tail_frames=tail_frames,
            show_static_objects=show_static_objects,
            show_dynamic_objects=show_dynamic_objects,
            show_head=show_head,
            show_body=show_body,
            show_skeleton=show_skeleton,
            show_hit=show_hit,
            vertical_axis=vertical_axis,
            mirror_horizontal=mirror_horizontal,
        )
        scene_frame_data.append(traces)

    diag_rows = diagnostic_rows_for_frames(window, animation_frames)
    diag_static, angle_cache = diagnostic_static_traces(diag_rows, context.head, enabled_tracks)
    pitch_range = clipped_angle_axis_range(angle_cache, 0, enabled_tracks, (-90.0, 90.0))
    yaw_range = clipped_angle_axis_range(angle_cache, 1, enabled_tracks, (-180.0, 180.0))
    first_markers = diagnostic_marker_traces(diag_rows, animation_frames[0], enabled_tracks, angle_cache)

    fig = make_subplots(
        rows=3,
        cols=2,
        specs=[[{'type': 'scene', 'colspan': 2}, None], [{'type': 'xy'}, {'type': 'xy'}], [{'type': 'xy'}, {'type': 'xy'}]],
        subplot_titles=('Scene 3D scanpath', 'Angular error / MAE [deg]', 'Euclidean point distance [m]', 'CPF pitch [deg]', 'CPF yaw [deg]'),
        row_heights=[0.62, 0.19, 0.19],
        vertical_spacing=0.07,
        horizontal_spacing=0.08,
    )

    scene_trace_indices = []
    for trace in scene_frame_data[0]:
        fig.add_trace(trace, row=1, col=1)
        scene_trace_indices.append(len(fig.data) - 1)

    # Error traces.
    for trace in diag_static[0:2]:
        fig.add_trace(trace, row=2, col=1)
    for trace in diag_static[2:4]:
        fig.add_trace(trace, row=2, col=2)
    for trace in diag_static[4:7]:
        fig.add_trace(trace, row=3, col=1)
    for trace in diag_static[7:10]:
        fig.add_trace(trace, row=3, col=2)

    marker_trace_indices = []
    for trace in first_markers[0:2]:
        fig.add_trace(trace, row=2, col=1)
        marker_trace_indices.append(len(fig.data) - 1)
    for trace in first_markers[2:4]:
        fig.add_trace(trace, row=2, col=2)
        marker_trace_indices.append(len(fig.data) - 1)
    for trace in first_markers[4:7]:
        fig.add_trace(trace, row=3, col=1)
        marker_trace_indices.append(len(fig.data) - 1)
    for trace in first_markers[7:10]:
        fig.add_trace(trace, row=3, col=2)
        marker_trace_indices.append(len(fig.data) - 1)

    update_indices = scene_trace_indices + marker_trace_indices
    plotly_frames = []
    for frame_index, scene_traces in zip(animation_frames, scene_frame_data):
        marker_traces = diagnostic_marker_traces(diag_rows, int(frame_index), enabled_tracks, angle_cache)
        plotly_frames.append(go.Frame(
            name=str(frame_index),
            data=scene_traces + marker_traces,
            traces=update_indices,
            layout=go.Layout(title=f'{sequence} | frame={frame_index} | window=[{start_frame}, {end_frame}) | tail={tail_frames}'),
        ))
    fig.frames = plotly_frames

    ranges = axis_ranges_for_animation(scene_frame_data)
    x_label, y_label, z_label = scene04.display_axis_labels(vertical_axis, mirror_horizontal)
    sliders = [{
        'active': 0,
        'currentvalue': {'prefix': 'frame: '},
        'pad': {'t': 40},
        'steps': [
            {'args': [[str(frame_index)], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}], 'label': str(frame_index), 'method': 'animate'}
            for frame_index in animation_frames
        ],
    }]
    fig.update_layout(
        title=f'{sequence} | frame={animation_frames[0]} | window=[{start_frame}, {end_frame}) | tail={tail_frames}',
        scene=dict(
            xaxis=dict(title=f'{x_label} [m]', range=ranges[0]),
            yaxis=dict(title=f'{y_label} [m]', range=ranges[1]),
            zaxis=dict(title=f'{z_label} [m]', range=ranges[2]),
            aspectmode='data',
            camera=scene04.plotly_camera(vertical_axis, view_elev, view_azim),
        ),
        xaxis=dict(title='Frame'),
        yaxis=dict(title='Angular error [deg]'),
        xaxis2=dict(title='Frame'),
        yaxis2=dict(title='Euclidean distance [m]'),
        xaxis3=dict(title='Frame'),
        yaxis3=dict(title='CPF pitch [deg]', range=pitch_range),
        xaxis4=dict(title='Frame'),
        yaxis4=dict(title='CPF yaw [deg]', range=yaw_range),
        height=1050,
        legend=dict(x=1.02, y=1.0),
        margin=dict(l=0, r=160, t=70, b=0),
        updatemenus=[{
            'type': 'buttons',
            'direction': 'left',
            'x': 0.08,
            'y': 0,
            'pad': {'r': 10, 't': 70},
            'showactive': False,
            'buttons': [
                {'label': 'Play', 'method': 'animate', 'args': [None, {'frame': {'duration': int(frame_duration_ms), 'redraw': True}, 'fromcurrent': True, 'transition': {'duration': 0}}]},
                {'label': 'Pause', 'method': 'animate', 'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate', 'transition': {'duration': 0}}]},
            ],
        }],
        sliders=sliders,
    )
    return fig, animation_frames, tracks

In [5]:
sequences = list_sequences()
if not sequences:
    raise RuntimeError(f'No ADT sequences found under {REPORTS_DIR / "sequences"}')

default_sequence = 'Apartment_release_decoration_skeleton_seq133_M1292'
if default_sequence not in sequences:
    default_sequence = sequences[0]

sequence_dropdown = widgets.Dropdown(options=sequences, value=default_sequence, description='sequence', layout=widgets.Layout(width='760px'))
start_input = widgets.BoundedIntText(value=149, min=0, max=100000, description='start', layout=widgets.Layout(width='180px'))
end_input = widgets.BoundedIntText(value=300, min=1, max=100000, description='end', layout=widgets.Layout(width='180px'))
frame_stride_input = widgets.BoundedIntText(value=5, min=1, max=200, description='frame stride', layout=widgets.Layout(width='190px'))
point_stride_input = widgets.BoundedIntText(value=1, min=1, max=100, description='point stride', layout=widgets.Layout(width='190px'))
tail_input = widgets.BoundedIntText(value=30, min=0, max=5000, description='tail', layout=widgets.Layout(width='160px'))
max_frames_input = widgets.BoundedIntText(value=80, min=1, max=500, description='max frames', layout=widgets.Layout(width='190px'))
frame_duration_input = widgets.BoundedIntText(value=140, min=20, max=2000, description='ms/frame', layout=widgets.Layout(width='180px'))
max_objects_input = widgets.BoundedIntText(value=80, min=0, max=1000, description='objects', layout=widgets.Layout(width='180px'))
tracks_select = widgets.SelectMultiple(options=TRACK_LABELS, value=tuple(TRACK_LABELS), description='tracks', layout=widgets.Layout(width='260px', height='90px'))
show_static_checkbox = widgets.Checkbox(value=True, description='static boxes', indent=False)
show_dynamic_checkbox = widgets.Checkbox(value=True, description='dynamic boxes', indent=False)
show_head_checkbox = widgets.Checkbox(value=True, description='head traj', indent=False)
show_body_checkbox = widgets.Checkbox(value=True, description='body traj', indent=False)
show_skeleton_checkbox = widgets.Checkbox(value=True, description='skeleton', indent=False)
show_hit_checkbox = widgets.Checkbox(value=True, description='hit point', indent=False)
mirror_checkbox = widgets.Checkbox(value=True, description='mirror X', indent=False)
vertical_dropdown = widgets.Dropdown(options=['scene_y', 'scene_z', 'scene_x'], value='scene_y', description='up')
view_elev = widgets.FloatSlider(value=22.0, min=-90.0, max=90.0, step=1.0, description='elev', continuous_update=False)
view_azim = widgets.FloatSlider(value=-58.0, min=-180.0, max=180.0, step=1.0, description='azim', continuous_update=False)
build_button = widgets.Button(description='Build animation', button_style='primary', icon='play')
status = widgets.HTML()
out = widgets.Output()


def sync_frame_bounds(*_):
    try:
        _, _, table = load_sequence_data(sequence_dropdown.value)
        max_frame = int(table['frame_index'].max())
        start_input.max = max_frame
        end_input.max = max_frame + 1
        if end_input.value > max_frame + 1:
            end_input.value = max_frame + 1
        status.value = f'<span style="color:#555">loaded {sequence_dropdown.value}: {max_frame + 1} frames</span>'
    except Exception as exc:
        status.value = f'<span style="color:#b91c1c">failed to load sequence: {exc}</span>'


def build(_=None):
    with out:
        clear_output(wait=True)
        try:
            status.value = '<span style="color:#555">building animation...</span>'
            fig, frame_indices, loaded_tracks = build_animation_scene_figure(
                sequence=sequence_dropdown.value,
                start_frame=start_input.value,
                end_frame=end_input.value,
                frame_stride=frame_stride_input.value,
                point_stride=point_stride_input.value,
                tail_frames=tail_input.value,
                max_animation_frames=max_frames_input.value,
                enabled_tracks=tuple(tracks_select.value),
                show_static_objects=show_static_checkbox.value,
                show_dynamic_objects=show_dynamic_checkbox.value,
                show_head=show_head_checkbox.value,
                show_body=show_body_checkbox.value,
                show_skeleton=show_skeleton_checkbox.value,
                show_hit=show_hit_checkbox.value,
                max_static_objects=max_objects_input.value,
                vertical_axis=vertical_dropdown.value,
                mirror_horizontal=mirror_checkbox.value,
                view_elev=view_elev.value,
                view_azim=view_azim.value,
                frame_duration_ms=frame_duration_input.value,
            )
            display(fig)
            status.value = f'<span style="color:#166534">built {len(frame_indices)} animation frames | tracks: {", ".join(loaded_tracks)}</span>'
        except Exception as exc:
            status.value = f'<span style="color:#b91c1c">build failed: {exc}</span>'
            raise

sequence_dropdown.observe(sync_frame_bounds, names='value')
build_button.on_click(build)
sync_frame_bounds()

controls = widgets.VBox([
    sequence_dropdown,
    widgets.HBox([start_input, end_input, frame_stride_input, point_stride_input, tail_input]),
    widgets.HBox([max_frames_input, frame_duration_input, max_objects_input]),
    widgets.HBox([tracks_select, widgets.VBox([show_static_checkbox, show_dynamic_checkbox, show_head_checkbox, show_body_checkbox, show_skeleton_checkbox, show_hit_checkbox])]),
    widgets.HBox([vertical_dropdown, mirror_checkbox, view_elev, view_azim, build_button]),
    status,
])

display(controls, out)
build()


Output()

Notes:

- Click **Build animation** after changing controls. The resulting Plotly figure has its own Play/Pause buttons and frame slider.
- `frame stride` controls which frames become animation frames.
- `point stride` only downsamples points drawn inside each frame's trajectory/tail.
- `max frames` prevents accidental construction of very large Plotly animations.
- Plotly keeps the scene interactive during pause: rotate, zoom, and pan with the mouse.